## Stage 1 - ResNet50 Baseline

### Data folder structure 

In [1]:
import os

for root, dirs, files in os.walk("JIGSAWS"):
    level = root.replace("JIGSAWS", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"    {indent}{f}")

JIGSAWS/
    .DS_Store
  Experimental_setup/
      .DS_Store
    Knot_Tying/
        .DS_Store
      Balanced/
        GestureClassification/
          OneTrialOut/
            25_Out/
              itr_44/
              itr_43/
              itr_17/
              itr_28/
              itr_10/
              itr_9/
              itr_26/
              itr_19/
              itr_21/
              itr_7/
              itr_42/
              itr_45/
              itr_6/
              itr_20/
              itr_27/
              itr_1/
              itr_18/
              itr_11/
              itr_8/
              itr_16/
              itr_29/
              itr_34/
              itr_33/
              itr_32/
              itr_35/
              itr_50/
              itr_13/
              itr_14/
              itr_22/
              itr_4/
              itr_3/
              itr_25/
              itr_49/
              itr_40/
              itr_47/
              itr_24/
              itr_2/
         

In [2]:
print(os.listdir("Cholec80"))
print(os.listdir("JIGSAWS"))

['.DS_Store', 'cholec80']
['.DS_Store', 'Experimental_setup', 'Knot_Tying', 'Suturing', 'Needle_Passing']


In [3]:
with open("JIGSAWS/Suturing/transcriptions/Suturing_B001.txt") as f:
    print(f.read())

80 219 G1 
220 370 G5 
371 590 G8 
591 660 G2 
661 954 G3 
955 1097 G8 
1098 1124 G2 
1125 1401 G3 
1402 1439 G2 
1440 1698 G8 
1699 1783 G2 
1784 2285 G3 
2286 2495 G6 
2496 2679 G4 
2680 2750 G2 
2751 2976 G3 
2977 3155 G6 
3156 3308 G4 
3309 3659 G2 
3660 4011 G3 
4012 4321 G8 
4322 4374 G2 
4375 4704 G3 
4705 4846 G6 
4847 4983 G4 
4984 5059 G2 
5060 5354 G3 
5355 5447 G6 
5448 5635 G11 



In [4]:
videos = os.listdir("JIGSAWS/Suturing/video/")
print(sorted(videos)[:5])
print(f"Total videos: {len(videos)}")

['Suturing_B001_capture1.avi', 'Suturing_B001_capture2.avi', 'Suturing_B002_capture1.avi', 'Suturing_B002_capture2.avi', 'Suturing_B003_capture1.avi']
Total videos: 78


### Dataloader

In [5]:
import torch 
from torch.utils.data import Dataset, DataLoader
import av
import numpy as np 
import os
from torchvision import transforms
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# mapping gestures to numbers
GESTURE_MAP = {
    'G1':0,'G2':1,'G3':2,'G4':3,'G5':4,
    'G6':5,'G8':6,'G9':7,'G10':8,'G11':9
}

In [6]:
class JIGSAWSDataset(Dataset):
    def __init__(self, task='Suturing', split_trials=None, clip_len=8):
        # split_trials is the list of trial names to include like 'B001', 'B002'

        self.clip_len = clip_len
        self.clips = [] # will (store video_path, start, end, label)

        trans_dir = f"JIGSAWS/{task}/transcriptions/"
        video_dir = f"JIGSAWS/{task}/video/"

        # looping through every transcription file - one per video
        for fname in sorted(os.listdir(trans_dir)):
            if not fname.endswith('.txt'):
                continue

            # extract trial name
            trial = fname.replace(f'{task}_', '').replace('.txt', '')

            # test trials skipped
            if split_trials and trial not in split_trials:
                continue
            
            # we use capture 1 camera angle
            video_path = os.path.join(video_dir, f"{task}_{trial}_capture1.avi") 
            if not os.path.exists(video_path):
                continue

            # read transcription file
            with open(os.path.join(trans_dir, fname)) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 3: continue
                    start, end, gesture = int(parts[0]), int(parts[1]), parts[2]

                    # keeping only gestures we know and long enough to sample 8 frames
                    if gesture in GESTURE_MAP and (end-start)>clip_len:
                        self.clips.append((video_path, start, end, GESTURE_MAP[gesture]))

                
        # transform to match ResNet
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias = True),
            transforms.Normalize([0.485,0.456,0.406],
                                [0.229,0.224,0.225]) 
        ])
        print(f"    {task} {split_trials}: {len(self.clips)} gesture clips")

    
    def _load_frames(self, video_path, start, end):
        # pick clip_len frame indices evenly spread across the gesture segment

        frame_ids = set(np.linspace(start, end, self.clip_len, dtype=int).tolist())
        frames = []
        container = av.open(video_path) # open the video file
        for i, frame in enumerate(container.decode(video=0)):
            if i in frame_ids:
                img = frame.to_ndarray(format='rgb24') # converting to RGB numpy array
                frames.append(self.transform(img)) # resize + transform
            if len(frames) == self.clip_len:
                break # stop when we have all frames

        container.close()

        while len(frames) < self.clip_len:
            frames.append(frames[-1]) # pad the last frame if less

        return torch.stack(frames) # stach into (T, C, H, W) tensor

    def __len__(self): return len(self.clips)

    def __getitem__(self,idx):
        video_path, start, end, label = self.clips[idx]
        frames = self._load_frames(video_path, start, end)

        # for stage 1 we throw away all but the middle frame
        # naive baseline - one photo, no motion info

        mid = frames[self.clip_len // 2] # (C, H, W)

        return mid, label            

In [7]:
# loading a sample to check

ds = JIGSAWSDataset(task="Suturing")
img, label = ds[0]
print(f"Sample shape: {img.shape}, label: {label}, gesture: {list(GESTURE_MAP.keys())[label]}")

    Suturing None: 793 gesture clips
Sample shape: torch.Size([3, 224, 224]), label: 0, gesture: G1


### Train/test split

In [8]:
# get all trial names from the transciptions folder
all_trials = [f.replace('Suturing_', '').replace('.txt','')
            for f in os.listdir('JIGSAWS/Suturing/transcriptions/')
            if f.endswith('.txt')]
all_trials = sorted(all_trials)
print("All trials:", all_trials)

# first four trials for test rest train
# JIGSAWS uses leave-one-user-out i am doing it in later stages

test_trials = all_trials[:4]
train_trials = all_trials[4:]

print(f"Train: {train_trials}")
print(f"Test: {test_trials}")

#  create dataset objects
from torch.utils.data import ConcatDataset

tasks = ['Suturing', 'Knot_Tying', 'Needle_Passing']

#concat all 3 datasets to one
train_ds = ConcatDataset([JIGSAWSDataset(t, split_trials=train_trials) for t in tasks])
test_ds  = ConcatDataset([JIGSAWSDataset(t, split_trials=test_trials)  for t in tasks])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)

print(f"Total train clips: {len(train_ds)}")
print(f"Total test clips:  {len(test_ds)}")
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

All trials: ['B001', 'B002', 'B003', 'B004', 'B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Train: ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Test: ['B001', 'B002', 'B003', 'B004']
    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Knot_Tying ['B005', 'C001', 'C0

### ResNet50 Model

In [9]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# load resnet50
model = resnet50(weights = ResNet50_Weights.DEFAULT)

# replace final layer as we only have 10 classes and imagenet has 1000
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device) 

# Adam optimizer - adapts lr per param
# weight decay adds L2 regularization to prevent overfitting
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# halving the lr every 5 epochs - helps fine convergence later
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {total_params:,}")
print(f"Device: {device}")

Trainable params: 23,528,522
Device: mps


### Training Loop on all tasks together

In [10]:
from sklearn.metrics import f1_score
import time

def run_epoch(loader, train=True):
    model.train() if train else model.eval() # train mode vs eval mode (effects dropout)
    total_loss, all_preds, all_labels = 0, [], []

    with torch.set_grad_enabled(train): # dont compute grad during eval
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            out = model(imgs) # forward pass
            loss = criterion(out, labels) # cross entropy loss

            if train:
                optimizer.zero_grad() # clear grads from last batch 
                loss.backward()  # compute gradient via backdrop
                optimizer.step() # update weights

            total_loss += loss.item()
            all_preds.extend(out.argmax(1).cpu().numpy()) # predicted class per sample
            all_labels.extend(labels.cpu().numpy()) # true class per sample

    
        # macro F1 - all 10 gestures equal
        f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        return total_loss/len(loader), f1

print("Starting Stage 1 training - Resnet50 single-frame baseline")
print("="*50)

best_test_f1 = 0

for epoch in range(1, 16):
    t0 = time.time()
    tr_loss, tr_f1 = run_epoch(train_loader, train=True)
    te_loss, te_f1 = run_epoch(test_loader, train=False)
    scheduler.step()  # reduce lr if scheduled

    if te_f1 > best_test_f1: 
        best_test_f1 = te_f1

    print(f"Epoch {epoch:2d} | "
          f"Train loss: {tr_loss:.3f} F1: {tr_f1:.3f} | "
          f"Test  loss: {te_loss:.3f} F1: {te_f1:.3f} | "
          f"{time.time()-t0:.1f}s")

print(f"\nBest Test F1: {best_test_f1:.3f}")

Starting Stage 1 training - Resnet50 single-frame baseline
Epoch  1 | Train loss: 1.947 F1: 0.177 | Test  loss: 1.613 F1: 0.273 | 165.7s
Epoch  2 | Train loss: 1.146 F1: 0.418 | Test  loss: 1.285 F1: 0.433 | 144.1s
Epoch  3 | Train loss: 0.528 F1: 0.684 | Test  loss: 1.067 F1: 0.615 | 136.6s
Epoch  4 | Train loss: 0.237 F1: 0.827 | Test  loss: 1.136 F1: 0.561 | 136.7s
Epoch  5 | Train loss: 0.100 F1: 0.868 | Test  loss: 1.068 F1: 0.651 | 136.7s
Epoch  6 | Train loss: 0.044 F1: 0.891 | Test  loss: 1.111 F1: 0.607 | 136.6s
Epoch  7 | Train loss: 0.030 F1: 0.896 | Test  loss: 1.177 F1: 0.600 | 136.3s
Epoch  8 | Train loss: 0.023 F1: 0.972 | Test  loss: 1.160 F1: 0.609 | 137.0s
Epoch  9 | Train loss: 0.019 F1: 0.986 | Test  loss: 1.218 F1: 0.588 | 137.5s
Epoch 10 | Train loss: 0.016 F1: 0.988 | Test  loss: 1.223 F1: 0.622 | 137.2s
Epoch 11 | Train loss: 0.012 F1: 0.988 | Test  loss: 1.262 F1: 0.573 | 137.0s
Epoch 12 | Train loss: 0.011 F1: 1.000 | Test  loss: 1.215 F1: 0.623 | 137.3s
Epoch

### Training Loop per task

In [11]:
print("Stage 1 - Per task training and eval")
print("="*50)

tasks = ['Suturing', 'Knot_Tying', 'Needle_Passing']
task_results = {}

for task in tasks:
    print(f"\n--- {task} ---")

    # prep dataset like before
    task_train_ds = JIGSAWSDataset(task, split_trials=train_trials)
    task_test_ds = JIGSAWSDataset(task, split_trials=test_trials)

    if len(task_test_ds) == 0:
        print(f"    Skipping - no test clips")
        continue

    task_train_loader = DataLoader(task_train_ds, batch_size=32, shuffle=True, num_workers=0)
    task_test_loader  = DataLoader(task_test_ds,  batch_size=32, shuffle=False, num_workers=0)

    # new model for each task
    task_model = resnet50(weights=ResNet50_Weights.DEFAULT)
    task_model.fc = nn.Linear(task_model.fc.in_features, 10)
    task_model = task_model.to(device)

    task_optimizer  = torch.optim.Adam(task_model.parameters(), lr=1e-4, weight_decay=1e-4)
    task_scheduler  = torch.optim.lr_scheduler.StepLR(task_optimizer, step_size=5, gamma=0.5)
    task_criterion  = nn.CrossEntropyLoss()

    best_f1 = 0
    for epoch in range(1,16):

        # train
        task_model.train()
        tr_preds, tr_labels = [], []
        for imgs, labels in task_train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = task_model(imgs)
            loss = task_criterion(out, labels)
            task_optimizer.zero_grad()
            loss.backward()
            task_optimizer.step()
            tr_preds.extend(out.argmax(1).cpu().numpy())
            tr_labels.extend(labels.cpu().numpy())
        task_scheduler.step()
        tr_f1 = f1_score(tr_labels, tr_preds, average='macro', zero_division=0)

        # evaluate
        task_model.eval()
        te_preds, te_labels = [], []
        with torch.no_grad():
            for imgs, labels in task_test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = task_model(imgs)
                te_preds.extend(out.argmax(1).cpu().numpy())
                te_labels.extend(labels.cpu().numpy())
        te_f1 = f1_score(te_labels, te_preds, average='macro', zero_division=0)

        if te_f1 > best_f1:
            best_f1 = te_f1

        print(f"  Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f}")

    task_results[task] = best_f1

# summary code
print("\n" + "="*50)
print("STAGE 1 SUMMARY — Single frame ResNet50")
print("="*60)
print(f"  Combined (all tasks): Test F1 = (from previous cell)")
for task, f1 in task_results.items():
    print(f"  {task:20s}: Best Test F1 = {f1:.3f}")

Stage 1 - Per task training and eval

--- Suturing ---
    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Suturing ['B001', 'B002', 'B003', 'B004']: 83 gesture clips
  Epoch  1 | Train F1: 0.157 | Test F1: 0.225
  Epoch  2 | Train F1: 0.306 | Test F1: 0.339
  Epoch  3 | Train F1: 0.618 | Test F1: 0.502
  Epoch  4 | Train F1: 0.828 | Test F1: 0.693
  Epoch  5 | Train F1: 0.878 | Test F1: 0.673
  Epoch  6 | Train F1: 0.936 | Test F1: 0.687
  Epoch  7 | Train F1: 0.937 | Test F1: 0.713
  Epoch  8 | Train F1: 1.000 | Test F1: 0.687
  Epoch  9 | Train F1: 0.964 | Test F1: 0.702
  Epoch 10 | Train F1: 0.985 | Test F1: 0.664
  Epoch 11 | Train F1: 0.998 | Test F1: 0.698
  Epoch 12 | Train F1: 1.000 | Test F1: 0.679
  Epoch 13 | Tra